# 04D1F_v3 — LF2019 Limited Candidate Disambiguation

**Build ID: `04D1F_V3_LF2019L_CANDIDATE_DISAMBIGUATION`**

04D1F V2 successfully resolved all five LF2020 ScienceBase archives.

The remaining issue is LF2019 Limited:

- 5 required products;
- 9 tied top archive candidates per product;
- 45 tied archive URLs total.

This notebook determines whether those 9 candidates/product are:

1. **multipart spatial archives** that must all be acquired;
2. **duplicate/mirror copies** of one archive;
3. a mixed/ambiguous set requiring manual review.

## Safety

This notebook does **not** download archive bodies.

It uses:

- the V2 candidate CSV already written locally;
- URL/path structure;
- HTTP `HEAD`;
- optional HTTP byte-range reads of ZIP metadata **only if the server
  explicitly honors Range requests**.

If a server ignores a Range request and responds with the full archive, the
response is closed immediately without consuming the body.

No LANDFIRE production feature is changed here.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1F_V3_LF2019L_CANDIDATE_DISAMBIGUATION"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

V2_OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_inventory_v2"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_inventory_v3"
)

CANDIDATES_PATH = (
    V2_OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_CANDIDATES_V2.csv"
)

DIRECTORY_INVENTORY_PATH = (
    V2_OUTPUT_ROOT /
    "LF2019L_HELPDESK_DIRECTORY_INVENTORY_V2.csv"
)

SELECTION_V2_PATH = (
    V2_OUTPUT_ROOT /
    "LANDFIRE_ARCHIVE_RECOMMENDED_SELECTION_V2.csv"
)

for p in [
    CANDIDATES_PATH,
    DIRECTORY_INVENTORY_PATH,
    SELECTION_V2_PATH,
]:
    assert p.exists(), p

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REQUIRED_PRODUCTS = [
    "FBFM40",
    "CC",
    "CH",
    "CBH",
    "CBD",
]

REQUEST_TIMEOUT_S = 90

# Strict partial-read limits.
ZIP_TAIL_PROBE_BYTES = 131_072       # 128 KiB
ZIP_CDIR_MAX_BYTES = 8 * 1024 * 1024 # 8 MiB
MAX_RANGE_RESPONSE_BYTES = (
    ZIP_CDIR_MAX_BYTES
    + 1024
)

# Explicitly no archive download.
ALLOW_FULL_ARCHIVE_DOWNLOAD = False

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("ALLOW_FULL_ARCHIVE_DOWNLOAD:", ALLOW_FULL_ARCHIVE_DOWNLOAD)

assert ALLOW_FULL_ARCHIVE_DOWNLOAD is False

## 1. Dependencies and session

In [ ]:
required = {
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

from datetime import datetime, timezone
from urllib.parse import urlparse, unquote
import hashlib
import json
import math
import re
import struct

import numpy as np
import pandas as pd
import requests

pd.set_option(
    "display.max_columns",
    250,
)
pd.set_option(
    "display.max_rows",
    1000,
)
pd.set_option(
    "display.width",
    420,
)

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "UCSD-wildfire-fireline-effectiveness/"
        "04D1F-v3-LF2019L-disambiguation"
    )
})

## 2. Load the V2 candidate inventory

In [ ]:
candidates = pd.read_csv(
    CANDIDATES_PATH
)

directory_inventory = pd.read_csv(
    DIRECTORY_INVENTORY_PATH
)

selection_v2 = pd.read_csv(
    SELECTION_V2_PATH
)

lf2019 = candidates[
    candidates[
        "version_key"
    ] == "LF2019L"
].copy()

lf2019 = lf2019[
    lf2019[
        "product"
    ].isin(
        REQUIRED_PRODUCTS
    )
].copy()

print(
    "LF2019L candidate rows:",
    f"{len(lf2019):,}",
)

summary = (
    lf2019.groupby(
        "product",
        as_index=False,
    )
    .agg(
        candidate_rows=(
            "candidate_url",
            "size",
        ),
        unique_urls=(
            "candidate_url",
            "nunique",
        ),
        unique_names=(
            "candidate_name",
            "nunique",
        ),
        max_score=(
            "candidate_score",
            "max",
        ),
    )
)

display(
    summary
)

assert set(
    summary["product"]
) == set(
    REQUIRED_PRODUCTS
)

assert (
    summary[
        "unique_urls"
    ] >= 1
).all()

## 3. Show all tied top candidates with URL-path context

In [ ]:
def url_parent_parts(url):
    p = urlparse(
        str(url)
    )

    parts = [
        unquote(x)
        for x in p.path.split("/")
        if x
    ]

    return {
        "url_host": p.netloc,
        "url_path": unquote(
            p.path
        ),
        "parent_dir": (
            parts[-2]
            if len(parts) >= 2
            else None
        ),
        "grandparent_dir": (
            parts[-3]
            if len(parts) >= 3
            else None
        ),
        "basename_from_url": (
            parts[-1]
            if parts
            else None
        ),
    }


path_rows = []

for _, r in lf2019.iterrows():
    meta = url_parent_parts(
        r["candidate_url"]
    )

    path_rows.append({
        **r.to_dict(),
        **meta,
    })

lf2019_paths = pd.DataFrame(
    path_rows
)

top_score_by_product = (
    lf2019_paths.groupby(
        "product"
    )[
        "candidate_score"
    ].transform(
        "max"
    )
)

tied_top = lf2019_paths[
    lf2019_paths[
        "candidate_score"
    ] == top_score_by_product
].copy()

tied_top = tied_top.sort_values(
    [
        "product",
        "candidate_name",
        "url_path",
    ]
).reset_index(
    drop=True
)

display(
    tied_top[
        [
            "product",
            "candidate_name",
            "candidate_score",
            "grandparent_dir",
            "parent_dir",
            "url_path",
        ]
    ]
)

print()
print(
    "Tied-top rows:",
    f"{len(tied_top):,}",
)

print(
    "Tied-top URLs:",
    f"{tied_top['candidate_url'].nunique():,}",
)

tied_top.to_csv(
    OUTPUT_ROOT /
    "LF2019L_TIED_TOP_CANDIDATES_V3.csv",
    index=False,
)

## 4. HEAD metadata for all tied candidates

Content length, ETag and Last-Modified can reveal obvious mirror duplicates.

They are **evidence**, not cryptographic proof of identity.

In [ ]:
def head_metadata(
    url,
):
    try:
        r = session.head(
            url,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT_S,
        )

        return {
            "head_status": r.status_code,
            "head_final_url": r.url,
            "content_length": pd.to_numeric(
                r.headers.get(
                    "Content-Length"
                ),
                errors="coerce",
            ),
            "content_type": r.headers.get(
                "Content-Type"
            ),
            "etag": r.headers.get(
                "ETag"
            ),
            "last_modified": r.headers.get(
                "Last-Modified"
            ),
            "accept_ranges": r.headers.get(
                "Accept-Ranges"
            ),
            "content_disposition": r.headers.get(
                "Content-Disposition"
            ),
            "head_error": None,
        }

    except Exception as exc:
        return {
            "head_status": np.nan,
            "head_final_url": None,
            "content_length": np.nan,
            "content_type": None,
            "etag": None,
            "last_modified": None,
            "accept_ranges": None,
            "content_disposition": None,
            "head_error": (
                f"{type(exc).__name__}: "
                f"{str(exc)[:1000]}"
            ),
        }


head_rows = []

for i, (_, r) in enumerate(
    tied_top.iterrows(),
    start=1,
):
    print(
        f"HEAD {i}/{len(tied_top)}:",
        r["product"],
        r["candidate_name"],
    )

    h = head_metadata(
        r["candidate_url"]
    )

    head_rows.append({
        **r.to_dict(),
        **h,
    })


head_table = pd.DataFrame(
    head_rows
)

display(
    head_table[
        [
            "product",
            "candidate_name",
            "parent_dir",
            "head_status",
            "content_length",
            "etag",
            "last_modified",
            "accept_ranges",
            "content_type",
        ]
    ]
)

head_table.to_csv(
    OUTPUT_ROOT /
    "LF2019L_TIED_TOP_HEAD_METADATA_V3.csv",
    index=False,
)

## 5. First-pass duplicate/partition signatures

Within each product we compare:

- URL basename;
- content length;
- ETag;
- Last-Modified;
- parent-directory path.

Interpretation:

- one repeated metadata signature across all 9 → likely mirrors/duplicates;
- 9 distinct names/sizes/signatures → likely multipart archives;
- mixed signatures → further ZIP metadata inspection.

In [ ]:
def clean_sig_value(v):
    if pd.isna(
        v
    ):
        return None

    return str(
        v
    )


signature_rows = []

for product, g in head_table.groupby(
    "product",
    sort=False,
):
    g = g.copy()

    g["metadata_signature"] = [
        json.dumps(
            {
                "name": clean_sig_value(
                    row[
                        "candidate_name"
                    ]
                ),
                "size": clean_sig_value(
                    row[
                        "content_length"
                    ]
                ),
                "etag": clean_sig_value(
                    row[
                        "etag"
                    ]
                ),
                "last_modified": clean_sig_value(
                    row[
                        "last_modified"
                    ]
                ),
            },
            sort_keys=True,
        )
        for _, row in g.iterrows()
    ]

    signature_rows.append({
        "product": product,
        "candidate_count": len(g),
        "unique_urls": g[
            "candidate_url"
        ].nunique(),
        "unique_names": g[
            "candidate_name"
        ].nunique(),
        "unique_sizes": g[
            "content_length"
        ].nunique(
            dropna=True
        ),
        "unique_etags": g[
            "etag"
        ].nunique(
            dropna=True
        ),
        "unique_last_modified": g[
            "last_modified"
        ].nunique(
            dropna=True
        ),
        "unique_metadata_signatures": g[
            "metadata_signature"
        ].nunique(),
        "all_head_200": bool(
            (
                g[
                    "head_status"
                ] == 200
            ).all()
        ),
    })


signature_summary = pd.DataFrame(
    signature_rows
)

display(
    signature_summary
)

signature_summary.to_csv(
    OUTPUT_ROOT /
    "LF2019L_CANDIDATE_SIGNATURE_SUMMARY_V3.csv",
    index=False,
)

## 6. Safe HTTP Range helper

A range probe is accepted only when the server returns HTTP `206` and a valid
`Content-Range`.

If the server responds `200` with a full archive, the response is immediately
closed and **zero archive bytes are consumed by the notebook code**.

In [ ]:
def safe_range_get(
    url,
    range_header,
    max_bytes,
):
    headers = {
        "Range": range_header,
    }

    try:
        r = session.get(
            url,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT_S,
        )

        status = r.status_code

        content_range = r.headers.get(
            "Content-Range"
        )

        if status != 206:
            r.close()

            return {
                "success": False,
                "status": status,
                "content_range": content_range,
                "data": None,
                "error": (
                    "server_did_not_honor_range"
                ),
            }

        if not content_range:
            r.close()

            return {
                "success": False,
                "status": status,
                "content_range": None,
                "data": None,
                "error": (
                    "206_without_content_range"
                ),
            }

        content_length = pd.to_numeric(
            r.headers.get(
                "Content-Length"
            ),
            errors="coerce",
        )

        if (
            pd.notna(
                content_length
            )
            and float(
                content_length
            ) > max_bytes
        ):
            r.close()

            return {
                "success": False,
                "status": status,
                "content_range": content_range,
                "data": None,
                "error": (
                    "range_response_exceeds_safety_limit"
                ),
            }

        chunks = []
        total = 0

        for chunk in r.iter_content(
            chunk_size=64 * 1024
        ):
            if not chunk:
                continue

            total += len(
                chunk
            )

            if total > max_bytes:
                r.close()

                return {
                    "success": False,
                    "status": status,
                    "content_range": content_range,
                    "data": None,
                    "error": (
                        "stream_exceeded_safety_limit"
                    ),
                }

            chunks.append(
                chunk
            )

        r.close()

        return {
            "success": True,
            "status": status,
            "content_range": content_range,
            "data": b"".join(
                chunks
            ),
            "error": None,
        }

    except Exception as exc:
        return {
            "success": False,
            "status": np.nan,
            "content_range": None,
            "data": None,
            "error": (
                f"{type(exc).__name__}: "
                f"{str(exc)[:1000]}"
            ),
        }


print(
    "SAFE RANGE HELPER: READY"
)

## 7. ZIP central-directory metadata probe

For ZIPs whose server supports HTTP Range:

1. fetch the final 128 KiB;
2. locate the End of Central Directory (EOCD);
3. read the central-directory offset/size;
4. fetch only that byte range if necessary;
5. parse central-directory filenames and member sizes.

This can identify geographic tiles/parts without downloading raster contents.

In [ ]:
EOCD_SIG = b"PK\x05\x06"
CDIR_SIG = b"PK\x01\x02"


def parse_eocd(
    tail_bytes,
):
    idx = tail_bytes.rfind(
        EOCD_SIG
    )

    if idx < 0:
        return None

    if len(
        tail_bytes
    ) < idx + 22:
        return None

    fields = struct.unpack_from(
        "<4s4H2LH",
        tail_bytes,
        idx,
    )

    (
        sig,
        disk_number,
        cdir_disk,
        entries_disk,
        entries_total,
        cdir_size,
        cdir_offset,
        comment_length,
    ) = fields

    return {
        "disk_number": disk_number,
        "cdir_disk": cdir_disk,
        "entries_disk": entries_disk,
        "entries_total": entries_total,
        "cdir_size": cdir_size,
        "cdir_offset": cdir_offset,
        "comment_length": comment_length,
    }


def parse_central_directory(
    data,
):
    members = []
    pos = 0

    while pos + 46 <= len(
        data
    ):
        if data[
            pos:pos + 4
        ] != CDIR_SIG:
            break

        (
            sig,
            version_made,
            version_needed,
            flags,
            compression,
            mod_time,
            mod_date,
            crc32,
            compressed_size,
            uncompressed_size,
            filename_len,
            extra_len,
            comment_len,
            disk_start,
            internal_attr,
            external_attr,
            local_header_offset,
        ) = struct.unpack_from(
            "<4s6H3L5H2L",
            data,
            pos,
        )

        start = (
            pos + 46
        )

        end = (
            start
            + filename_len
        )

        if end > len(
            data
        ):
            break

        raw_name = data[
            start:end
        ]

        encoding = (
            "utf-8"
            if (
                flags
                & 0x800
            )
            else "cp437"
        )

        try:
            name = raw_name.decode(
                encoding,
                errors="replace",
            )
        except Exception:
            name = repr(
                raw_name
            )

        members.append({
            "member_name": name,
            "compressed_size": compressed_size,
            "uncompressed_size": uncompressed_size,
            "crc32": (
                f"{crc32:08x}"
            ),
            "compression_method": compression,
            "local_header_offset": local_header_offset,
        })

        pos = (
            end
            + extra_len
            + comment_len
        )

    return members


def zip_metadata_probe(
    url,
    file_size,
):
    if pd.isna(
        file_size
    ):
        return {
            "probe_status": "NO_FILE_SIZE",
            "members": [],
            "error": None,
        }

    file_size = int(
        file_size
    )

    tail_start = max(
        0,
        file_size
        - ZIP_TAIL_PROBE_BYTES,
    )

    tail = safe_range_get(
        url,
        (
            f"bytes={tail_start}-"
            f"{file_size - 1}"
        ),
        ZIP_TAIL_PROBE_BYTES
        + 1024,
    )

    if not tail[
        "success"
    ]:
        return {
            "probe_status": "RANGE_UNSUPPORTED",
            "members": [],
            "error": tail[
                "error"
            ],
            "range_status": tail[
                "status"
            ],
        }

    eocd = parse_eocd(
        tail[
            "data"
        ]
    )

    if eocd is None:
        return {
            "probe_status": "EOCD_NOT_FOUND",
            "members": [],
            "error": None,
        }

    # ZIP64 archives cannot be reliably parsed from classic EOCD size fields
    # when they contain sentinel values. Record and stop conservatively.
    if (
        eocd[
            "entries_total"
        ] == 0xFFFF
        or eocd[
            "cdir_size"
        ] == 0xFFFFFFFF
        or eocd[
            "cdir_offset"
        ] == 0xFFFFFFFF
    ):
        return {
            "probe_status": "ZIP64_REVIEW_REQUIRED",
            "members": [],
            "error": None,
            **eocd,
        }

    cdir_size = int(
        eocd[
            "cdir_size"
        ]
    )

    cdir_offset = int(
        eocd[
            "cdir_offset"
        ]
    )

    if cdir_size > ZIP_CDIR_MAX_BYTES:
        return {
            "probe_status": "CDIR_TOO_LARGE_FOR_SAFE_PROBE",
            "members": [],
            "error": None,
            **eocd,
        }

    # Central directory may already be wholly inside the tail probe.
    relative_start = (
        cdir_offset
        - tail_start
    )

    if (
        relative_start >= 0
        and (
            relative_start
            + cdir_size
        ) <= len(
            tail[
                "data"
            ]
        )
    ):
        cdir_data = tail[
            "data"
        ][
            relative_start:
            relative_start
            + cdir_size
        ]

    else:
        cdir_end = (
            cdir_offset
            + cdir_size
            - 1
        )

        cdir = safe_range_get(
            url,
            (
                f"bytes={cdir_offset}-"
                f"{cdir_end}"
            ),
            ZIP_CDIR_MAX_BYTES
            + 1024,
        )

        if not cdir[
            "success"
        ]:
            return {
                "probe_status": "CDIR_RANGE_FAILED",
                "members": [],
                "error": cdir[
                    "error"
                ],
                **eocd,
            }

        cdir_data = cdir[
            "data"
        ]

    members = parse_central_directory(
        cdir_data
    )

    return {
        "probe_status": (
            "SUCCESS"
            if members
            else "CDIR_PARSE_EMPTY"
        ),
        "members": members,
        "error": None,
        **eocd,
    }

## 8. Probe tied LF2019L archives

In [ ]:
probe_summary_rows = []
member_rows = []

for i, (_, r) in enumerate(
    head_table.iterrows(),
    start=1,
):
    print(
        f"ZIP META {i}/{len(head_table)}:",
        r["product"],
        r["candidate_name"],
    )

    probe = zip_metadata_probe(
        r[
            "candidate_url"
        ],
        r[
            "content_length"
        ],
    )

    members = probe.pop(
        "members",
        [],
    )

    probe_summary_rows.append({
        "product": r[
            "product"
        ],
        "candidate_name": r[
            "candidate_name"
        ],
        "candidate_url": r[
            "candidate_url"
        ],
        "parent_dir": r[
            "parent_dir"
        ],
        "content_length": r[
            "content_length"
        ],
        "etag": r[
            "etag"
        ],
        **probe,
        "member_count_parsed": len(
            members
        ),
    })

    for m in members:
        member_rows.append({
            "product": r[
                "product"
            ],
            "candidate_name": r[
                "candidate_name"
            ],
            "candidate_url": r[
                "candidate_url"
            ],
            **m,
        })


probe_summary = pd.DataFrame(
    probe_summary_rows
)

zip_members = pd.DataFrame(
    member_rows
)

display(
    probe_summary[
        [
            "product",
            "candidate_name",
            "parent_dir",
            "content_length",
            "probe_status",
            "member_count_parsed",
        ]
    ]
)

print()
print(
    "ZIP member rows parsed:",
    f"{len(zip_members):,}",
)

if len(
    zip_members
):
    display(
        zip_members[
            [
                "product",
                "candidate_name",
                "member_name",
                "compressed_size",
                "uncompressed_size",
            ]
        ].head(
            500
        )
    )

probe_summary.to_csv(
    OUTPUT_ROOT /
    "LF2019L_ZIP_METADATA_PROBE_SUMMARY_V3.csv",
    index=False,
)

zip_members.to_csv(
    OUTPUT_ROOT /
    "LF2019L_ZIP_MEMBER_INVENTORY_V3.csv",
    index=False,
)

## 9. Geographic/part token extraction

We look for evidence that the tied archives form a deliberate multi-part set.

The classifier does not assume any specific tiling convention. It records
tokens that vary across candidate basenames, directories and internal GeoTIFF
member names.

In [ ]:
def stem_tokens(
    text,
):
    if pd.isna(
        text
    ):
        return []

    stem = Path(
        str(
            text
        )
    ).stem

    return [
        t
        for t in re.split(
            r"[^A-Za-z0-9]+",
            stem,
        )
        if t
    ]


def varying_tokens(
    values,
):
    token_sets = [
        set(
            stem_tokens(
                v
            )
        )
        for v in values
    ]

    if not token_sets:
        return []

    common = set.intersection(
        *token_sets
    )

    variable = sorted(
        set.union(
            *token_sets
        )
        - common
    )

    return variable


part_rows = []

for product, g in head_table.groupby(
    "product",
    sort=False,
):
    member_g = (
        zip_members[
            zip_members[
                "product"
            ] == product
        ]
        if len(
            zip_members
        )
        else pd.DataFrame()
    )

    part_rows.append({
        "product": product,
        "candidate_count": len(g),
        "unique_candidate_names": g[
            "candidate_name"
        ].nunique(),
        "unique_parent_dirs": g[
            "parent_dir"
        ].nunique(
            dropna=True
        ),
        "variable_filename_tokens": json.dumps(
            varying_tokens(
                g[
                    "candidate_name"
                ].tolist()
            )
        ),
        "variable_parent_tokens": json.dumps(
            varying_tokens(
                g[
                    "parent_dir"
                ].dropna().tolist()
            )
        ),
        "zip_probe_success_count": int(
            (
                probe_summary.loc[
                    probe_summary[
                        "product"
                    ] == product,
                    "probe_status",
                ]
                == "SUCCESS"
            ).sum()
        ),
        "unique_internal_tif_names": (
            int(
                member_g.loc[
                    member_g[
                        "member_name"
                    ].str.lower().str.endswith(
                        (
                            ".tif",
                            ".tiff",
                        ),
                        na=False,
                    ),
                    "member_name",
                ].nunique()
            )
            if len(
                member_g
            )
            else 0
        ),
    })


part_evidence = pd.DataFrame(
    part_rows
)

display(
    part_evidence
)

part_evidence.to_csv(
    OUTPUT_ROOT /
    "LF2019L_MULTIPART_EVIDENCE_V3.csv",
    index=False,
)

## 10. Conservative classification

The notebook classifies each product only when the evidence is strong.

### `LIKELY_MIRROR_DUPLICATES`

All tied URLs share one candidate basename and one non-null content-length
signature, with no evidence of distinct internal GeoTIFF names.

### `LIKELY_MULTIPART_SET`

There are multiple distinct archive names or distinct sizes/ETags, and the
candidate set is not reducible to one metadata signature.

### `AMBIGUOUS_REVIEW`

Evidence is insufficient or contradictory.

A multipart classification means the next downloader must acquire **all
members of the set**, not choose a single archive.

In [ ]:
classification_rows = []

for product, g in head_table.groupby(
    "product",
    sort=False,
):
    sig = signature_summary[
        signature_summary[
            "product"
        ] == product
    ].iloc[
        0
    ]

    ps = probe_summary[
        probe_summary[
            "product"
        ] == product
    ].copy()

    mg = (
        zip_members[
            zip_members[
                "product"
            ] == product
        ].copy()
        if len(
            zip_members
        )
        else pd.DataFrame()
    )

    distinct_tifs = (
        mg.loc[
            mg[
                "member_name"
            ].str.lower().str.endswith(
                (
                    ".tif",
                    ".tiff",
                ),
                na=False,
            ),
            "member_name",
        ].nunique()
        if len(
            mg
        )
        else 0
    )

    unique_sizes = int(
        g[
            "content_length"
        ].nunique(
            dropna=True
        )
    )

    unique_etags = int(
        g[
            "etag"
        ].nunique(
            dropna=True
        )
    )

    unique_names = int(
        g[
            "candidate_name"
        ].nunique()
    )

    unique_urls = int(
        g[
            "candidate_url"
        ].nunique()
    )

    successful_probes = int(
        (
            ps[
                "probe_status"
            ] == "SUCCESS"
        ).sum()
    )

    # Strong duplicate evidence.
    likely_mirror = bool(
        unique_names == 1
        and unique_sizes <= 1
        and (
            unique_etags <= 1
        )
        and distinct_tifs <= 1
    )

    # Strong multipart evidence.
    likely_multipart = bool(
        unique_urls > 1
        and (
            unique_names > 1
            or unique_sizes > 1
            or unique_etags > 1
            or distinct_tifs > 1
        )
    )

    if likely_mirror and not likely_multipart:
        status = (
            "LIKELY_MIRROR_DUPLICATES"
        )

    elif likely_multipart and not likely_mirror:
        status = (
            "LIKELY_MULTIPART_SET"
        )

    else:
        status = (
            "AMBIGUOUS_REVIEW"
        )

    classification_rows.append({
        "product": product,
        "classification": status,
        "candidate_count": len(g),
        "unique_urls": unique_urls,
        "unique_names": unique_names,
        "unique_sizes": unique_sizes,
        "unique_etags": unique_etags,
        "successful_zip_metadata_probes": successful_probes,
        "unique_internal_tif_names": int(
            distinct_tifs
        ),
        "download_plan_if_accepted": (
            "ALL_TIED_ARCHIVES"
            if status
            == "LIKELY_MULTIPART_SET"
            else (
                "ONE_VERIFIED_COPY"
                if status
                == "LIKELY_MIRROR_DUPLICATES"
                else "NONE_PENDING_REVIEW"
            )
        ),
    })


classification = pd.DataFrame(
    classification_rows
)

display(
    classification
)

classification.to_csv(
    OUTPUT_ROOT /
    "LF2019L_CANDIDATE_CLASSIFICATION_V3.csv",
    index=False,
)

## 11. Build LF2019L acquisition-set proposal

In [ ]:
plan_rows = []

for product in REQUIRED_PRODUCTS:

    c = classification[
        classification[
            "product"
        ] == product
    ].iloc[
        0
    ]

    g = head_table[
        head_table[
            "product"
        ] == product
    ].copy()

    if (
        c[
            "classification"
        ]
        == "LIKELY_MULTIPART_SET"
    ):
        selected = g.copy()

        plan_status = (
            "PROPOSE_ALL_PARTS"
        )

    elif (
        c[
            "classification"
        ]
        == "LIKELY_MIRROR_DUPLICATES"
    ):
        selected = g.sort_values(
            [
                "head_status",
                "candidate_url",
            ],
            ascending=[
                False,
                True,
            ],
        ).head(
            1
        )

        plan_status = (
            "PROPOSE_ONE_MIRROR"
        )

    else:
        selected = g.iloc[
            0:0
        ].copy()

        plan_status = (
            "NO_AUTOMATIC_SELECTION"
        )

    if selected.empty:
        plan_rows.append({
            "version_key": "LF2019L",
            "product": product,
            "plan_status": plan_status,
            "candidate_name": None,
            "candidate_url": None,
            "content_length": np.nan,
            "etag": None,
            "download": False,
        })

    else:
        for _, r in selected.iterrows():
            plan_rows.append({
                "version_key": "LF2019L",
                "product": product,
                "plan_status": plan_status,
                "candidate_name": r[
                    "candidate_name"
                ],
                "candidate_url": r[
                    "candidate_url"
                ],
                "content_length": r[
                    "content_length"
                ],
                "etag": r[
                    "etag"
                ],
                "download": True,
            })


acquisition_plan = pd.DataFrame(
    plan_rows
)

display(
    acquisition_plan
)

print()
print(
    "Proposed LF2019L archive count:",
    int(
        acquisition_plan[
            "download"
        ].sum()
    ),
)

print(
    "Proposed total bytes:",
    int(
        acquisition_plan.loc[
            acquisition_plan[
                "download"
            ],
            "content_length",
        ]
        .fillna(
            0
        )
        .sum()
    ),
)

acquisition_plan.to_csv(
    OUTPUT_ROOT /
    "LF2019L_ACQUISITION_SET_PROPOSAL_V3.csv",
    index=False,
)

## 12. Final checks and decision

In [ ]:
checks = []

checks.append({
    "check": (
        "all_five_products_classified"
    ),
    "pass": (
        len(
            classification
        ) == 5
        and set(
            classification[
                "product"
            ]
        )
        == set(
            REQUIRED_PRODUCTS
        )
    ),
})

checks.append({
    "check": (
        "no_full_archive_download_enabled"
    ),
    "pass": (
        ALLOW_FULL_ARCHIVE_DOWNLOAD
        is False
    ),
})

checks.append({
    "check": (
        "all_tied_candidates_received_HEAD"
    ),
    "pass": bool(
        head_table[
            "head_status"
        ].notna().all()
    ),
})

checks.append({
    "check": (
        "no_product_left_ambiguous"
    ),
    "pass": bool(
        (
            classification[
                "classification"
            ]
            != "AMBIGUOUS_REVIEW"
        ).all()
    ),
})

checks_df = pd.DataFrame(
    checks
)

display(
    checks_df
)

READY_FOR_ACQUISITION_REVIEW = bool(
    checks_df[
        "pass"
    ].all()
)

print()
print(
    "04D1F V3 LF2019L DISAMBIGUATION:",
    (
        "PASS / READY TO REVIEW ACQUISITION SET"
        if READY_FOR_ACQUISITION_REVIEW
        else "REVIEW REQUIRED"
    ),
)

checks_df.to_csv(
    OUTPUT_ROOT /
    "LF2019L_DISAMBIGUATION_CHECKS_V3.csv",
    index=False,
)

## 13. Report

In [ ]:
def human_bytes(
    value,
):
    if pd.isna(
        value
    ):
        return None

    x = float(
        value
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:
        if x < 1024:
            return (
                f"{x:.2f} {unit}"
            )
        x /= 1024.0

    return (
        f"{x:.2f} PiB"
    )


plan_report = (
    acquisition_plan.copy()
)

plan_report[
    "size_human"
] = plan_report[
    "content_length"
].apply(
    human_bytes
)


def tomd(df):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


report = f"""# 04D1F V3 — LF2019 Limited Candidate Disambiguation

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Tied candidate summary

{tomd(summary)}

## Metadata signatures

{tomd(signature_summary)}

## ZIP metadata probe

{tomd(probe_summary)}

## Multipart evidence

{tomd(part_evidence)}

## Classification

{tomd(classification)}

## Proposed acquisition set

{tomd(plan_report)}

## Checks

{tomd(checks_df)}

## Decision

`{"PASS / READY TO REVIEW ACQUISITION SET" if READY_FOR_ACQUISITION_REVIEW else "REVIEW REQUIRED"}`

No full archive was downloaded by this notebook.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LF2019L_CANDIDATE_DISAMBIGUATION_REPORT_V3.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH,
)

## 14. Output inventory

In [ ]:
print("=" * 100)
print("04D1F V3 OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)